# Microproyecto 2: generación de código Python con Transformers

## 1. Portada e introducción

**Curso:** Modelos Avanzados de Procesamiento de Lenguaje Natural (MAIA, Universidad de los Andes)
**Equipo:** MP1 Equipo 21
**Integrantes:** Juan Valera · Juan David Villalba · Dayana Vega
**Dataset:** `codeparrot/codeparrot-clean` (obligatorio)

### Resumen

Se entrena y compara la capacidad de cinco configuraciones de modelo de lenguaje para completar código Python. Todas comparten el tokenizador BPE de SmolLM2 (vocabulario de 49.152 tokens) y una longitud de contexto fija de 256 tokens, de modo que la perplejidad es directamente comparable entre ellas.

| # | Configuración | Origen de los pesos | Entrenamiento |
|---|---|---|---|
| 1 | GPT desde cero (decoder-only, PyTorch) | inicialización aleatoria | 3 épocas, AdamW lr 3e-4 |
| 2 | SmolLM2-135M pre-entrenado | `HuggingFaceTB/SmolLM2-135M` | ninguno (baseline) |
| 3 | SmolLM2-135M finetuned | `HuggingFaceTB/SmolLM2-135M` | finetuning completo, 1 época, lr 5e-5 |
| 4 | SmolLM2-360M pre-entrenado | `HuggingFaceTB/SmolLM2-360M` | ninguno (baseline) |
| 5 | SmolLM2-360M finetuned | `HuggingFaceTB/SmolLM2-360M` | finetuning completo, 1 época, lr 3e-5 |

**Métricas sobre los 800 archivos de prueba:** perplejidad a nivel de token (métrica principal) y porcentaje de validez sintáctica con `ast.parse` sobre 200 prompts con semilla fija (métrica secundaria). Se complementan con las curvas de pérdida de entrenamiento/validación y con tres ejemplos de generación por cada modelo entrenado por el equipo. La generación se implementa con un bucle autorregresivo propio, sin `model.generate()`.

### Ejecución

> **Importante para Google Colab:** Si ejecutó previamente otro notebook o hubo celdas interrumpidas en la misma sesión, ejecute **Entorno de ejecución > Reiniciar sesión** (*Runtime > Restart session*) antes de comenzar para asegurarse de que los 15-16 GB de VRAM de la GPU estén completamente limpios.

- Entorno objetivo: Google Colab con GPU (T4 o superior), opción *Run All*.
- Si los checkpoints ya existen en `checkpoints/`, se reutilizan en lugar de reentrenar (`REENTRENAR = False`). Los resultados de prueba se guardan en `checkpoints/resultados_prueba.json` y también se reutilizan salvo que `REENTRENAR` o `REEVALUAR` sean `True`.
- Variable de entorno `MAPLN_SMOKE=1`: prueba rápida de extremo a extremo (3 actualizaciones por modelo, evaluación reducida) para verificar que el notebook corre sin errores.
- Tiempo aproximado en una T4 con entrenamiento completo: GPT desde cero ≈ 5 min, SmolLM2-135M ≈ 15 min, SmolLM2-360M ≈ 30 min, evaluación de las cinco configuraciones ≈ 25 min.

## 2. Librerías

Instalación de lo que falte, importación de todas las dependencias, semillas, selección de dispositivo y constantes del proyecto.

- **Precisión mixta:** fp16 con `GradScaler` en GPUs sin soporte bf16 (T4, capacidad 7.5) y bf16 en Ampere o superior. Los pesos maestros y los estados de AdamW permanecen en fp32.
- **Contrato de respuesta:** los pasos de orquestación (carga del corpus, entrenamiento, finetuning y evaluación) devuelven un `ServiceResponse(success, data, error)`; `unwrap()` propaga el error con su traza si el paso falla.

In [ ]:
import importlib.util
import os
import subprocess
import sys

EN_COLAB = 'google.colab' in sys.modules
SALTAR_PIP = os.environ.get('MAPLN_SKIP_PIP') == '1'

if EN_COLAB:
    # Liberar cualquier proceso zombi previo que mantenga bloqueada la GPU en Colab
    os.system('fuser -k -9 /dev/nvidia* 2>/dev/null')
    paquetes = [p for p in ('datasets', 'accelerate') if importlib.util.find_spec(p) is None]
else:
    requeridos = {'torch': 'torch>=2.3', 'transformers': 'transformers>=4.44', 'datasets': 'datasets>=2.20',
                  'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib'}
    paquetes = [pip for modulo, pip in requeridos.items() if importlib.util.find_spec(modulo) is None]

if SALTAR_PIP:
    print('MAPLN_SKIP_PIP=1: no se instala ningún paquete.')
elif not paquetes:
    print('No falta ningún paquete.')
else:
    resultado = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--disable-pip-version-check', *paquetes],
        capture_output=True, text=True)
    print(resultado.stdout[-1500:])
    if resultado.returncode != 0:
        raise RuntimeError('Falló la instalación con pip:\n' + resultado.stderr[-1500:])
    print('Instalado:', ', '.join(paquetes))

In [ ]:
import ast
import contextlib
import functools
import gc
import itertools
import json
import math
import os
import random
import sys
import time
import traceback
import warnings
from collections import Counter
from dataclasses import dataclass
from typing import Any, Callable, Dict, Iterator, List, Optional, Sequence, Tuple, Union

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import datasets
import transformers
from datasets import load_dataset
from IPython.display import Markdown, display
from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup

transformers.logging.set_verbosity_error()
EN_COLAB = 'google.colab' in sys.modules

#: Historia de entrenamiento: pasos, pérdidas de entrenamiento/validación, lr y metadatos.
Historia = Dict[str, Any]
#: Cualquiera de los cinco modelos: GPTDesdeCero o un AutoModelForCausalLM de Hugging Face.
ModeloLM = nn.Module


@dataclass(frozen=True)
class ServiceResponse:
    """Contrato estandarizado de respuesta de los pasos de orquestación.

    Attributes:
        success: Estado booleano de la operación.
        data: Carga útil (corpus, historia, métricas) o None si hubo error.
        error: Mensaje descriptivo del error o None si la operación fue exitosa.
    """

    success: bool
    data: Any = None
    error: Optional[str] = None

    @classmethod
    def ok(cls, data: Any) -> 'ServiceResponse':
        """Construye una respuesta exitosa.

        Args:
            data: Carga útil de la operación.

        Returns:
            ServiceResponse con success=True.
        """
        return cls(True, data, None)

    @classmethod
    def fail(cls, error: str) -> 'ServiceResponse':
        """Construye una respuesta fallida.

        Args:
            error: Descripción del error.

        Returns:
            ServiceResponse con success=False y data=None.
        """
        return cls(False, None, error)

    def unwrap(self) -> Any:
        """Devuelve la carga útil o detiene la ejecución si la operación falló.

        Returns:
            El campo ``data``.

        Raises:
            RuntimeError: Si ``success`` es False; el mensaje incluye la traza original.
        """
        if not self.success:
            raise RuntimeError(self.error)
        return self.data


def como_servicio(funcion: Callable[..., Any]) -> Callable[..., ServiceResponse]:
    """Decorador que envuelve el resultado (o la excepción) de un paso en un ServiceResponse.

    Args:
        funcion: Paso de orquestación a envolver.

    Returns:
        Función con la misma firma que devuelve ServiceResponse.
    """
    @functools.wraps(funcion)
    def envoltura(*args: Any, **kwargs: Any) -> ServiceResponse:
        try:
            return ServiceResponse.ok(funcion(*args, **kwargs))
        except Exception as error:  # se reporta cualquier fallo del paso con su traza
            return ServiceResponse.fail(f'{type(error).__name__}: {error}\n{traceback.format_exc(limit=6)}')
    return envoltura


def fijar_semillas(semilla: int) -> None:
    """Fija las semillas de random, NumPy y PyTorch (CPU y CUDA).

    Args:
        semilla: Valor de la semilla.
    """
    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(semilla)


def obtener_dispositivo() -> torch.device:
    """Verifica el acceso real al acelerador de hardware y recupera la GPU si quedó bloqueada."""
    if torch.cuda.is_available():
        try:
            t = torch.zeros(1, device='cuda')
            del t
            torch.cuda.empty_cache()
            return torch.device('cuda')
        except Exception as err:
            print(f'Advertencia: torch.cuda reporta disponible pero falló al acceder a la GPU ({err}).')
            if EN_COLAB:
                print('Intentando recuperar GPU liberando procesos bloqueados con fuser...')
                os.system('fuser -k -9 /dev/nvidia* 2>/dev/null')
                time.sleep(1)
                try:
                    t = torch.zeros(1, device='cuda')
                    del t
                    torch.cuda.empty_cache()
                    return torch.device('cuda')
                except Exception:
                    pass
            print('Se utilizará CPU como respaldo.')
            return torch.device('cpu')
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


DEVICE = obtener_dispositivo()

# Precisión mixta solo en CUDA. bf16 exige capacidad >= 8.0; la T4 (7.5) usa fp16 + GradScaler.
USAR_AMP: bool = DEVICE.type == 'cuda'
AMP_DTYPE: torch.dtype = (torch.bfloat16 if USAR_AMP and torch.cuda.get_device_capability()[0] >= 8
                          else torch.float16)
USAR_SCALER: bool = USAR_AMP and AMP_DTYPE == torch.float16


def contexto_amp(activo: bool = True) -> contextlib.AbstractContextManager:
    """Devuelve el contexto de precisión mixta (autocast) o un contexto nulo.

    Args:
        activo: Si es False se fuerza fp32 aunque haya GPU.

    Returns:
        ``torch.autocast`` en CUDA con ``AMP_DTYPE``; ``nullcontext`` en otro caso.
    """
    if USAR_AMP and activo:
        return torch.autocast(device_type='cuda', dtype=AMP_DTYPE)
    return contextlib.nullcontext()


def liberar_gpu() -> None:
    """Recolecta basura y devuelve al asignador de CUDA la memoria liberada."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


liberar_gpu()


print('Python       :', sys.version.split()[0])
print('torch        :', torch.__version__)
print('transformers :', transformers.__version__)
print('datasets     :', datasets.__version__)
print('numpy        :', np.__version__)
print('pandas       :', pd.__version__)
print('matplotlib   :', matplotlib.__version__)
print('Dispositivo  :', DEVICE)
if DEVICE.type == 'cuda':
    propiedades = torch.cuda.get_device_properties(0)
    print(f'GPU          : {propiedades.name} ({propiedades.total_memory / 1e9:.1f} GB, '
          f'capacidad {propiedades.major}.{propiedades.minor})')
precision = 'fp32 (sin precisión mixta)' if not USAR_AMP else str(AMP_DTYPE).replace('torch.', '')
print('Precisión    :', precision, '+ GradScaler' if USAR_SCALER else '')

In [ ]:
SEMILLA = 42

# Datos: límites obligatorios del enunciado (sección 3.2)
DATASET = 'codeparrot/codeparrot-clean'
MIN_CHARS, MAX_CHARS = 1000, 2000
N_EJEMPLOS = 8000
N_TRAIN, N_VAL, N_TEST = 6400, 800, 800
CONTEXTO = 256
N_EXPLORACION = 5000

# Modelos pre-entrenados obligatorios (comparten tokenizador)
MODELOS_HF = {'135M': 'HuggingFaceTB/SmolLM2-135M', '360M': 'HuggingFaceTB/SmolLM2-360M'}

# GPT desde cero (hiperparámetros recomendados: AdamW, lr ~3e-4, ~3 épocas)
GPT_CFG = dict(vocab=49152, contexto=CONTEXTO, d_modelo=384, n_cabezas=6, n_capas=6, dropout=0.1)
GPT_ENTRENO = dict(lr=3e-4, wd=0.1, betas=(0.9, 0.95), epocas=3, micro_batch=16, grad_accum=1)

# Finetuning completo con batch efectivo de 16 (micro-batch x acumulación de gradiente)
FT_ENTRENO = {
    '135M': dict(lr=5e-5, wd=0.01, betas=(0.9, 0.999), epocas=1, micro_batch=4, grad_accum=4),
    '360M': dict(lr=3e-5, wd=0.01, betas=(0.9, 0.999), epocas=1, micro_batch=2, grad_accum=8),
}

WARMUP = 0.05          # fracción de pasos de calentamiento lineal
CLIP = 1.0             # norma máxima del gradiente
EVALS_POR_EPOCA = 4    # evaluaciones de validación por época
LOTE_EVAL = 8          # bloques por lote en evaluación

# Validez sintáctica y generación
N_VALIDEZ = 200
PROMPT_TOKENS = 64
NUEVOS_TOKENS = 128
NUEVOS_EJEMPLO = 160
TEMPERATURA = 0.8
TOP_K = 50
LOTE_GEN = 50

# Interruptores
REENTRENAR = False     # True: entrena aunque existan checkpoints
REEVALUAR = False      # True: recalcula métricas de prueba aunque exista el JSON de resultados
USAR_DRIVE = False     # True en Colab: checkpoints persistentes en Google Drive

SMOKE = os.environ.get('MAPLN_SMOKE') == '1'
MAX_PASOS: Optional[int] = None
MAX_BLOQUES_EVAL: Optional[int] = None
if SMOKE:
    N_EXPLORACION, MAX_PASOS, MAX_BLOQUES_EVAL = 300, 3, 16
    N_VALIDEZ, NUEVOS_TOKENS, NUEVOS_EJEMPLO, LOTE_GEN = 8, 16, 16, 8
    print('*** MODO DE PRUEBA RÁPIDA: 3 actualizaciones por modelo y evaluación sobre 16 bloques ***')

if USAR_DRIVE and EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ = '/content/drive/MyDrive/MAPLN_Microproyecto_2'
else:
    RAIZ = '.'
DIR_DATOS = os.path.join(RAIZ, 'datos')
DIR_CKPT = os.path.join(RAIZ, 'checkpoints_smoke' if SMOKE else 'checkpoints')
RUTA_GPT = os.path.join(DIR_CKPT, 'gpt_desde_cero.pt')
CARPETA_FT = {clave: os.path.join(DIR_CKPT, f'smollm2_{clave.lower()}_ft') for clave in MODELOS_HF}
RUTA_RESULTADOS = os.path.join(DIR_CKPT, 'resultados_prueba.json')
os.makedirs(DIR_DATOS, exist_ok=True)
os.makedirs(DIR_CKPT, exist_ok=True)

fijar_semillas(SEMILLA)
print(f'Semilla {SEMILLA} | {N_EJEMPLOS} archivos de {MIN_CHARS}-{MAX_CHARS} caracteres | '
      f'split {N_TRAIN}/{N_VAL}/{N_TEST} | contexto {CONTEXTO} tokens')
print(f'Checkpoints en: {os.path.abspath(DIR_CKPT)} | reentrenar: {REENTRENAR} | reevaluar: {REEVALUAR}')

## 3. Carga del dataset y exploración

Lectura en modo *streaming* de una muestra cruda de 5.000 archivos de `codeparrot/codeparrot-clean` (sin descargar el dataset completo): campos disponibles, distribución de longitudes, licencias y un ejemplo de código dentro del rango obligatorio.

In [ ]:
stream = load_dataset(DATASET, split='train', streaming=True)

t0 = time.time()
muestra: List[Dict[str, Any]] = list(itertools.islice(stream, N_EXPLORACION))
print(f'{len(muestra)} archivos leídos en {time.time() - t0:.0f} s')
print('Campos de cada ejemplo:', list(muestra[0].keys()))
print('\nPrimer ejemplo (sin el código):')
for campo, valor in muestra[0].items():
    if campo != 'content':
        print(f'  {campo:>13}: {valor}')

In [ ]:
largos = np.array([len(e['content']) for e in muestra])
en_rango = (largos >= MIN_CHARS) & (largos <= MAX_CHARS)

print(f'Caracteres por archivo: mediana {np.median(largos):,.0f} | p10 {np.percentile(largos, 10):,.0f} | '
      f'p90 {np.percentile(largos, 90):,.0f} | máximo {largos.max():,}')
print(f'Dentro del rango {MIN_CHARS}-{MAX_CHARS}: {en_rango.mean():.1%} de la muestra')

fig, ax = plt.subplots(figsize=(9, 3.6))
bordes = np.logspace(np.log10(max(largos.min(), 1)), np.log10(largos.max()), 60)
ax.hist(largos, bins=bordes, color='#4C72B0', edgecolor='white')
ax.axvspan(MIN_CHARS, MAX_CHARS, color='#DD8452', alpha=0.3, label='rango conservado (1.000-2.000)')
ax.set_xscale('log')
ax.set_xlabel('caracteres por archivo (escala log)')
ax.set_ylabel('archivos')
ax.set_title(f'Longitud de {len(muestra):,} archivos crudos de codeparrot-clean')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
licencias = Counter(e.get('license', 'desconocida') for e in muestra)
print('Licencias más frecuentes en la muestra:')
for licencia, n in licencias.most_common(6):
    print(f'  {licencia:<14} {n / len(muestra):6.1%}')

ejemplo = next(e for e in muestra if MIN_CHARS <= len(e['content']) <= MAX_CHARS)
print(f"\nEjemplo dentro del rango: {ejemplo.get('repo_name', '?')}/{ejemplo.get('path', '?')} "
      f"({len(ejemplo['content'])} caracteres)\n")
print(ejemplo['content'][:700])

## 4. Preprocesamiento

1. **Filtro de longitud** (1.000–2.000 caracteres) sobre el *streaming* hasta reunir exactamente 8.000 archivos. El corpus se guarda en `datos/corpus_codeparrot.json` para que las reejecuciones sean deterministas y no dependan de la red.
2. **Tokenización** con el tokenizador de SmolLM2, idéntico en las versiones de 135M y 360M (se verifica). Cada archivo termina con el token `EOS` para marcar el límite entre documentos.
3. **Segmentación** en bloques de 256 tokens, concatenando documentos y sin relleno (*padding*). Se aplica en la sección 5, después del split, para que ningún archivo quede repartido entre dos particiones.

In [ ]:
RUTA_CORPUS = os.path.join(DIR_DATOS, 'corpus_codeparrot.json')


@como_servicio
def cargar_corpus(ruta: str = RUTA_CORPUS) -> Tuple[List[str], int]:
    """Obtiene los N_EJEMPLOS primeros archivos del streaming que cumplen el filtro de longitud.

    Si existe la caché en disco se lee de ahí; si no, se recorre el streaming y se guarda.

    Args:
        ruta: Ruta del JSON de caché del corpus.

    Returns:
        Tupla (corpus, revisados): lista de códigos fuente y número de archivos recorridos.

    Raises:
        AssertionError: Si el corpus no cumple el tamaño o el filtro obligatorio.
    """
    if os.path.exists(ruta):
        with open(ruta, encoding='utf-8') as f:
            guardado = json.load(f)
        print(f'Corpus leído de {ruta} (caché).')
        corpus, revisados = guardado['corpus'], guardado['revisados']
    else:
        t0 = time.time()
        corpus, revisados = [], 0
        for registro in load_dataset(DATASET, split='train', streaming=True):
            revisados += 1
            codigo = registro['content']
            if not (MIN_CHARS <= len(codigo) <= MAX_CHARS):
                continue
            corpus.append(codigo)
            if len(corpus) >= N_EJEMPLOS:
                break
        with open(ruta, 'w', encoding='utf-8') as f:
            json.dump({'revisados': revisados, 'corpus': corpus}, f)
        print(f'Streaming terminado en {time.time() - t0:.0f} s.')
    assert len(corpus) == N_EJEMPLOS, f'Se esperaban {N_EJEMPLOS} archivos y hay {len(corpus)}'
    assert all(MIN_CHARS <= len(c) <= MAX_CHARS for c in corpus)
    return corpus, revisados


corpus, revisados = cargar_corpus().unwrap()
print(f'Archivos revisados: {revisados:,} | conservados: {len(corpus):,} ({len(corpus) / revisados:.1%})')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODELOS_HF['135M'])
tokenizer_360 = AutoTokenizer.from_pretrained(MODELOS_HF['360M'])

assert tokenizer.get_vocab() == tokenizer_360.get_vocab(), 'Los dos tokenizadores deberían ser idénticos'
assert tokenizer.vocab_size == GPT_CFG['vocab'] == 49152
EOS_ID: int = tokenizer.eos_token_id
del tokenizer_360

print('Tamaño del vocabulario (135M = 360M):', tokenizer.vocab_size)
print('Token de fin de texto:', repr(tokenizer.eos_token), '-> id', EOS_ID)

fragmento = 'def suma(a, b):\n    return a + b\n'
ids = tokenizer(fragmento, add_special_tokens=False)['input_ids']
print(f'\n{len(fragmento)} caracteres -> {len(ids)} tokens')
print(tokenizer.convert_ids_to_tokens(ids))

In [ ]:
t0 = time.time()
ids_docs: List[List[int]] = tokenizer(corpus, add_special_tokens=False)['input_ids']
ids_docs = [ids + [EOS_ID] for ids in ids_docs]
n_tokens = np.array([len(ids) for ids in ids_docs])
chars_por_token = sum(len(c) for c in corpus) / (n_tokens.sum() - len(ids_docs))

assert max(max(ids) for ids in ids_docs) < tokenizer.vocab_size
print(f'Tokenización: {time.time() - t0:.1f} s')
print(f'Tokens por archivo: mediana {np.median(n_tokens):.0f} | mínimo {n_tokens.min()} | '
      f'máximo {n_tokens.max()} | total {n_tokens.sum():,}')
print(f'Caracteres por token: {chars_por_token:.2f}')
print(f'Archivos más largos que una ventana de {CONTEXTO} tokens: {(n_tokens > CONTEXTO).mean():.0%}')

fig, ejes = plt.subplots(1, 2, figsize=(11, 3.4))
ejes[0].hist([len(c) for c in corpus], bins=40, color='#4C72B0', edgecolor='white')
ejes[0].set_xlabel('caracteres por archivo')
ejes[0].set_ylabel('archivos')
ejes[1].hist(n_tokens, bins=40, color='#55A868', edgecolor='white')
ejes[1].axvline(CONTEXTO, color='#C44E52', ls='--', label=f'ventana de {CONTEXTO} tokens')
ejes[1].set_xlabel('tokens por archivo (con EOS)')
ejes[1].legend()
fig.suptitle(f'Corpus filtrado: {len(corpus):,} archivos')
plt.tight_layout()
plt.show()

In [ ]:
def segmentar(lista_ids: Sequence[List[int]], contexto: int = CONTEXTO) -> torch.Tensor:
    """Concatena documentos tokenizados y los corta en bloques contiguos de longitud fija.

    Cada documento ya termina en EOS, de modo que el límite entre archivos queda marcado.
    Los tokens sobrantes del final (menos de un bloque) se descartan; no se usa relleno.

    Args:
        lista_ids: Documentos tokenizados (listas de ids).
        contexto: Longitud de cada bloque en tokens.

    Returns:
        Tensor int64. Shape: (n_bloques, contexto).
    """
    flujo = np.fromiter(itertools.chain.from_iterable(lista_ids), dtype=np.int64)
    n_bloques = len(flujo) // contexto
    return torch.from_numpy(flujo[:n_bloques * contexto].reshape(n_bloques, contexto))


demo = segmentar(ids_docs[:3])
total_demo = sum(len(x) for x in ids_docs[:3])
print(f'3 archivos = {total_demo} tokens -> {demo.shape[0]} bloques de {demo.shape[1]} '
      f'(se descartan {total_demo - demo.numel()} tokens del final)')

fila, columna = divmod(len(ids_docs[0]) - 1, CONTEXTO)
print(f'\nEl primer archivo termina en el bloque {fila}, posición {columna}. Texto alrededor del corte:')
print(repr(tokenizer.decode(demo[fila, max(columna - 12, 0):columna + 12].tolist())))

## 5. Split train / validación / prueba

Reparto por archivo completo, con permutación de semilla fija: 6.400 / 800 / 800 (80 / 10 / 10). Validación se usa para el monitoreo y la selección del mejor checkpoint; prueba se reserva exclusivamente para el reporte final de la sección 8.

In [ ]:
orden = np.random.default_rng(SEMILLA).permutation(N_EJEMPLOS)
idx_train = orden[:N_TRAIN]
idx_val = orden[N_TRAIN:N_TRAIN + N_VAL]
idx_test = orden[N_TRAIN + N_VAL:]

assert (len(idx_train), len(idx_val), len(idx_test)) == (N_TRAIN, N_VAL, N_TEST)
assert not (set(idx_train) & set(idx_val) or set(idx_train) & set(idx_test) or set(idx_val) & set(idx_test))

docs_train = [ids_docs[i] for i in idx_train]
docs_val = [ids_docs[i] for i in idx_val]
docs_test = [ids_docs[i] for i in idx_test]
print(f'Archivos: entrenamiento {len(docs_train)} | validación {len(docs_val)} | prueba {len(docs_test)}')

In [ ]:
bloques_train = segmentar(docs_train)
bloques_val = segmentar(docs_val)
bloques_test = segmentar(docs_test)

resumen_split = pd.DataFrame({
    'archivos': [len(docs_train), len(docs_val), len(docs_test)],
    'tokens': [sum(map(len, d)) for d in (docs_train, docs_val, docs_test)],
    'bloques de 256': [len(b) for b in (bloques_train, bloques_val, bloques_test)],
}, index=['entrenamiento', 'validación', 'prueba'])
resumen_split['tokens evaluados (255 por bloque)'] = resumen_split['bloques de 256'] * (CONTEXTO - 1)
display(resumen_split)

## 6. Modelo desde cero

GPT decoder-only escrito en PyTorch sin `nn.Transformer`: atención causal multi-cabeza implementada a mano, 6 bloques *pre-LayerNorm* con $d_{model} = 384$ y 6 cabezas, MLP $384 \to 1536 \to 384$ con GELU, embeddings de posición aprendidos y pesos atados entre el embedding de entrada y la capa de salida (≈ 29,6 M de parámetros, de los cuales 18,9 M corresponden al embedding de 49.152 × 384).

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}} + M\right)V, \qquad M_{ij} = \begin{cases} 0 & j \le i \\ -\infty & j > i \end{cases}$$

**Entrenamiento:** AdamW (lr 3e-4, $\beta = (0.9, 0.95)$, weight decay 0,1 solo en matrices), 3 épocas, batch 16, dropout 0,1, calentamiento lineal del 5 % y decaimiento coseno, recorte de gradiente a norma 1,0 y validación 4 veces por época. `entrenar()` es el mismo bucle que usan los dos finetunings y restaura los pesos del mejor paso de validación (*early stopping* por selección de checkpoint).

In [ ]:
class AtencionCausalMultiCabeza(nn.Module):
    """Atención multi-cabeza con máscara causal, implementada desde cero.

    Args:
        d_modelo: Dimensión del modelo (debe ser divisible por n_cabezas).
        n_cabezas: Número de cabezas de atención.
        contexto: Longitud máxima de secuencia (tamaño de la máscara).
        dropout: Probabilidad de dropout sobre los pesos de atención y la salida.
    """

    def __init__(self, d_modelo: int, n_cabezas: int, contexto: int, dropout: float) -> None:
        super().__init__()
        assert d_modelo % n_cabezas == 0
        self.n_cabezas = n_cabezas
        self.d_cabeza = d_modelo // n_cabezas
        self.qkv = nn.Linear(d_modelo, 3 * d_modelo)
        self.proyeccion = nn.Linear(d_modelo, d_modelo)
        self.drop_atencion = nn.Dropout(dropout)
        self.drop_salida = nn.Dropout(dropout)
        mascara = torch.tril(torch.ones(contexto, contexto, dtype=torch.bool))
        self.register_buffer('mascara', mascara.view(1, 1, contexto, contexto), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Aplica la atención causal.

        Args:
            x: Representaciones de entrada. Shape: (batch_size, seq_len, d_modelo).

        Returns:
            Salida de la atención. Shape: (batch_size, seq_len, d_modelo).
        """
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        # (B, T, C) -> (B, n_cabezas, T, d_cabeza)
        q = q.view(B, T, self.n_cabezas, self.d_cabeza).transpose(1, 2)
        k = k.view(B, T, self.n_cabezas, self.d_cabeza).transpose(1, 2)
        v = v.view(B, T, self.n_cabezas, self.d_cabeza).transpose(1, 2)

        puntajes = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_cabeza)          # (B, h, T, T)
        puntajes = puntajes.masked_fill(~self.mascara[:, :, :T, :T], float('-inf'))
        pesos = self.drop_atencion(F.softmax(puntajes, dim=-1))
        y = pesos @ v                                                            # (B, h, T, d_cabeza)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.drop_salida(self.proyeccion(y))

In [ ]:
class MLP(nn.Module):
    """Red feed-forward posición a posición: d -> 4d -> d con GELU.

    Args:
        d_modelo: Dimensión del modelo.
        dropout: Probabilidad de dropout sobre la salida.
    """

    def __init__(self, d_modelo: int, dropout: float) -> None:
        super().__init__()
        self.entrada = nn.Linear(d_modelo, 4 * d_modelo)
        self.salida = nn.Linear(4 * d_modelo, d_modelo)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Aplica la MLP.

        Args:
            x: Shape: (batch_size, seq_len, d_modelo).

        Returns:
            Shape: (batch_size, seq_len, d_modelo).
        """
        return self.drop(self.salida(F.gelu(self.entrada(x))))


class BloqueTransformer(nn.Module):
    """Bloque decoder pre-LN: x + Atención(LN(x)), luego x + MLP(LN(x)).

    Args:
        d_modelo: Dimensión del modelo.
        n_cabezas: Número de cabezas de atención.
        contexto: Longitud máxima de secuencia.
        dropout: Probabilidad de dropout.
    """

    def __init__(self, d_modelo: int, n_cabezas: int, contexto: int, dropout: float) -> None:
        super().__init__()
        self.ln_atencion = nn.LayerNorm(d_modelo)
        self.atencion = AtencionCausalMultiCabeza(d_modelo, n_cabezas, contexto, dropout)
        self.ln_mlp = nn.LayerNorm(d_modelo)
        self.mlp = MLP(d_modelo, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Aplica el bloque con conexiones residuales.

        Args:
            x: Shape: (batch_size, seq_len, d_modelo).

        Returns:
            Shape: (batch_size, seq_len, d_modelo).
        """
        x = x + self.atencion(self.ln_atencion(x))
        x = x + self.mlp(self.ln_mlp(x))
        return x

In [ ]:
class GPTDesdeCero(nn.Module):
    """Modelo de lenguaje decoder-only tipo GPT.

    Args:
        vocab: Tamaño del vocabulario (49.152, el de SmolLM2).
        contexto: Longitud máxima de secuencia (256).
        d_modelo: Dimensión de los embeddings y del flujo residual.
        n_cabezas: Cabezas de atención por bloque.
        n_capas: Número de bloques Transformer.
        dropout: Probabilidad de dropout.
    """

    def __init__(self, vocab: int, contexto: int, d_modelo: int, n_cabezas: int,
                 n_capas: int, dropout: float) -> None:
        super().__init__()
        self.contexto = contexto
        self.emb_tokens = nn.Embedding(vocab, d_modelo)
        self.emb_posicion = nn.Embedding(contexto, d_modelo)
        self.drop = nn.Dropout(dropout)
        self.bloques = nn.ModuleList(
            [BloqueTransformer(d_modelo, n_cabezas, contexto, dropout) for _ in range(n_capas)])
        self.ln_final = nn.LayerNorm(d_modelo)
        self.cabeza = nn.Linear(d_modelo, vocab, bias=False)
        self.cabeza.weight = self.emb_tokens.weight  # pesos atados entrada/salida

        self.apply(self._iniciar_pesos)
        # Escalado de las proyecciones residuales (GPT-2): estabiliza la varianza con la profundidad.
        for nombre, p in self.named_parameters():
            if nombre.endswith('proyeccion.weight') or nombre.endswith('salida.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_capas))

    @staticmethod
    def _iniciar_pesos(modulo: nn.Module) -> None:
        """Inicialización normal(0, 0.02) para capas lineales y embeddings; sesgos en cero.

        Args:
            modulo: Submódulo a inicializar.
        """
        if isinstance(modulo, nn.Linear):
            nn.init.normal_(modulo.weight, mean=0.0, std=0.02)
            if modulo.bias is not None:
                nn.init.zeros_(modulo.bias)
        elif isinstance(modulo, nn.Embedding):
            nn.init.normal_(modulo.weight, mean=0.0, std=0.02)

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        """Calcula los logits del siguiente token en cada posición.

        Args:
            ids: Ids de tokens. Shape: (batch_size, seq_len), con seq_len <= contexto.

        Returns:
            Logits. Shape: (batch_size, seq_len, vocab).

        Raises:
            AssertionError: Si seq_len supera el contexto del modelo.
        """
        B, T = ids.shape
        assert T <= self.contexto, f'La secuencia ({T}) supera el contexto ({self.contexto})'
        posiciones = torch.arange(T, device=ids.device)
        x = self.drop(self.emb_tokens(ids) + self.emb_posicion(posiciones))
        for bloque in self.bloques:
            x = bloque(x)
        return self.cabeza(self.ln_final(x))

In [ ]:
def logits_de(modelo: ModeloLM, x: torch.Tensor) -> torch.Tensor:
    """Obtiene los logits de cualquiera de las cinco configuraciones.

    Args:
        modelo: GPTDesdeCero (devuelve un tensor) o modelo de Hugging Face (devuelve un objeto con ``.logits``).
        x: Ids de entrada. Shape: (batch_size, seq_len).

    Returns:
        Logits. Shape: (batch_size, seq_len, vocab).
    """
    salida = modelo(x)
    return salida if torch.is_tensor(salida) else salida.logits


def nll_por_token(logits: torch.Tensor, x: torch.Tensor) -> Tuple[torch.Tensor, int]:
    """Suma de -log p(x_i | x_<i) sobre el lote y número de tokens predichos.

    La posición t predice el token t+1, por lo que cada bloque aporta seq_len - 1 predicciones.
    El cálculo se hace en fp32 para evitar pérdida de precisión en el log-softmax.

    Args:
        logits: Shape: (batch_size, seq_len, vocab).
        x: Ids verdaderos. Shape: (batch_size, seq_len).

    Returns:
        Tupla (suma_nll, n_tokens): suma escalar de la NLL y cantidad de tokens evaluados.
    """
    prediccion = logits[:, :-1, :].float()
    objetivo = x[:, 1:]
    suma = F.cross_entropy(prediccion.reshape(-1, prediccion.size(-1)), objetivo.reshape(-1), reduction='sum')
    return suma, objetivo.numel()


def iterar_lotes(bloques: torch.Tensor, tam: int, mezclar: bool = False,
                 generador: Optional[torch.Generator] = None) -> Iterator[torch.Tensor]:
    """Recorre los bloques en lotes, opcionalmente en orden aleatorio reproducible.

    Args:
        bloques: Shape: (n_bloques, seq_len).
        tam: Tamaño de lote.
        mezclar: Si es True se permuta el orden con ``generador``.
        generador: Generador de PyTorch para reproducibilidad.

    Yields:
        Lotes. Shape: (<=tam, seq_len).
    """
    orden = torch.randperm(len(bloques), generator=generador) if mezclar else torch.arange(len(bloques))
    for i in range(0, len(orden), tam):
        yield bloques[orden[i:i + tam]]


@torch.no_grad()
def evaluar_nll(modelo: ModeloLM, bloques: torch.Tensor, usar_amp: bool = True,
                tam_lote: int = LOTE_EVAL) -> float:
    """NLL promedio por token: acumula sumas y conteos y divide una sola vez al final.

    No se promedian pérdidas por lote; como no hay relleno, todos los tokens cuentan.

    Args:
        modelo: Modelo a evaluar.
        bloques: Shape: (n_bloques, seq_len).
        usar_amp: Si es False la pasada hacia adelante se hace en fp32.
        tam_lote: Bloques por lote.

    Returns:
        NLL promedio por token, en nats.
    """
    modelo.eval()
    if MAX_BLOQUES_EVAL:
        bloques = bloques[:MAX_BLOQUES_EVAL]
    suma, n = 0.0, 0
    for x in iterar_lotes(bloques, tam_lote):
        x = x.to(DEVICE)
        with contexto_amp(usar_amp):
            s, k = nll_por_token(logits_de(modelo, x), x)
        suma += s.item()
        n += k
    return suma / n


def contar_parametros(modelo: nn.Module) -> int:
    """Número total de parámetros (los pesos atados se cuentan una sola vez).

    Args:
        modelo: Modelo de PyTorch.

    Returns:
        Cantidad de parámetros.
    """
    return sum(p.numel() for p in modelo.parameters())

In [ ]:
fijar_semillas(SEMILLA)
liberar_gpu()
# La prueba de sanidad de la inicialización se evalúa en CPU para reservar el 100% de la VRAM al entrenamiento real
gpt_prueba = GPTDesdeCero(**GPT_CFG)
x = bloques_train[:2]
with torch.no_grad():
    logits = gpt_prueba(x)
    suma, n = nll_por_token(logits, x)

print('Entrada:', tuple(x.shape), '-> logits:', tuple(logits.shape))
print(f'Parámetros: {contar_parametros(gpt_prueba):,} '
      f'(de ellos {gpt_prueba.emb_tokens.weight.numel():,} en el embedding compartido con la salida)')
print(f'Pérdida inicial: {(suma / n).item():.3f} nats | ln({GPT_CFG["vocab"]}) = {math.log(GPT_CFG["vocab"]):.3f} '
      '(una predicción uniforme daría exactamente ln|V|)')
del gpt_prueba, logits, x
liberar_gpu()

In [ ]:
def entrenar(modelo: ModeloLM, nombre: str, lr: float, wd: float, betas: Tuple[float, float],
             epocas: int, micro_batch: int, grad_accum: int) -> Historia:
    """Bucle común de entrenamiento con AMP, acumulación de gradiente y selección del mejor checkpoint.

    Al terminar, ``modelo`` queda con los pesos del paso con menor pérdida de validación.

    Args:
        modelo: Modelo a entrenar (se mueve a DEVICE).
        nombre: Etiqueta para los registros y las gráficas.
        lr: Tasa de aprendizaje máxima de AdamW.
        wd: Weight decay (solo sobre parámetros con dim >= 2).
        betas: Coeficientes beta de AdamW.
        epocas: Épocas sobre ``bloques_train``.
        micro_batch: Bloques por pasada hacia adelante.
        grad_accum: Micro-lotes acumulados por actualización (batch efectivo = micro_batch * grad_accum).

    Returns:
        Historia con pasos, pérdidas de entrenamiento y validación, lr, mejor paso, tiempo y memoria.
    """
    liberar_gpu()
    modelo.to(DEVICE)
    con_decay = [p for p in modelo.parameters() if p.requires_grad and p.dim() >= 2]
    sin_decay = [p for p in modelo.parameters() if p.requires_grad and p.dim() < 2]
    optimizador = torch.optim.AdamW([{'params': con_decay, 'weight_decay': wd},
                                     {'params': sin_decay, 'weight_decay': 0.0}], lr=lr, betas=betas)

    pasos_por_epoca = (len(bloques_train) // micro_batch) // grad_accum
    pasos_totales = pasos_por_epoca * epocas
    if MAX_PASOS:
        pasos_totales = min(pasos_totales, MAX_PASOS)
    planificador = get_cosine_schedule_with_warmup(optimizador, max(1, int(WARMUP * pasos_totales)), pasos_totales)
    escalador = torch.amp.GradScaler('cuda', enabled=USAR_SCALER)
    cada = max(1, pasos_por_epoca // EVALS_POR_EPOCA)
    generador = torch.Generator().manual_seed(SEMILLA)

    historia: Historia = dict(nombre=nombre, batch_efectivo=micro_batch * grad_accum,
                              pasos=[0], train=[None], val=[evaluar_nll(modelo, bloques_val)], lr=[0.0])
    print(f'--- {nombre} | {contar_parametros(modelo) / 1e6:.1f} M parámetros | {pasos_totales} actualizaciones '
          f'(micro-batch {micro_batch} x acumulación {grad_accum}) ---')
    print(f'paso     0 | val {historia["val"][0]:.4f} (punto de partida)')

    mejor_val, mejor_paso = historia['val'][0], 0
    mejor_estado = {k: v.detach().to('cpu', copy=True) for k, v in modelo.state_dict().items()}
    suma_tr, tokens_tr, descartados, paso = 0.0, 0, 0, 0
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    modelo.train()
    optimizador.zero_grad(set_to_none=True)

    for epoca in range(epocas):
        for i, x in enumerate(iterar_lotes(bloques_train, micro_batch, mezclar=True, generador=generador)):
            if i >= pasos_por_epoca * grad_accum or paso >= pasos_totales:
                break
            x = x.to(DEVICE)
            with contexto_amp():
                suma, n = nll_por_token(logits_de(modelo, x), x)
            if torch.isfinite(suma):
                # Se divide por grad_accum para que el gradiente acumulado equivalga al de un lote grande.
                escalador.scale(suma / n / grad_accum).backward()
                suma_tr += suma.item()
                tokens_tr += n
            else:
                descartados += 1

            if (i + 1) % grad_accum == 0:
                escalador.unscale_(optimizador)
                nn.utils.clip_grad_norm_(modelo.parameters(), CLIP)
                escalador.step(optimizador)
                escalador.update()
                optimizador.zero_grad(set_to_none=True)
                planificador.step()
                paso += 1

                if paso % cada == 0 or paso == pasos_totales:
                    val = evaluar_nll(modelo, bloques_val)
                    modelo.train()
                    historia['pasos'].append(paso)
                    historia['train'].append(suma_tr / max(tokens_tr, 1))
                    historia['val'].append(val)
                    historia['lr'].append(planificador.get_last_lr()[0])
                    marca = ''
                    if val < mejor_val:
                        mejor_val, mejor_paso = val, paso
                        mejor_estado = {k: v.detach().to('cpu', copy=True) for k, v in modelo.state_dict().items()}
                        marca = '  <- mejor'
                    print(f'paso {paso:5d} | época {epoca + 1} | train {historia["train"][-1]:.4f} | val {val:.4f} '
                          f'| lr {historia["lr"][-1]:.1e} | {time.time() - t0:5.0f} s{marca}')
                    suma_tr, tokens_tr = 0.0, 0

    modelo.load_state_dict(mejor_estado)
    del optimizador, escalador, mejor_estado
    memoria = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else None
    historia.update(mejor_paso=mejor_paso, mejor_val=mejor_val, segundos=time.time() - t0,
                    descartados=descartados, memoria_max_gb=memoria)
    print(f'Mejor paso: {mejor_paso} (val {mejor_val:.4f}, PPL {math.exp(mejor_val):.2f}) | '
          f'{historia["segundos"] / 60:.1f} min'
          + (f' | memoria máxima {memoria:.1f} GB' if memoria else '')
          + (f' | lotes descartados por desborde fp16: {descartados}' if descartados else ''))
    return historia


def a_float(valores: Sequence[Optional[float]]) -> np.ndarray:
    """Convierte una lista con None a un arreglo float con NaN.

    Args:
        valores: Lista de valores o None.

    Returns:
        Arreglo float. Shape: (len(valores),).
    """
    return np.array([np.nan if v is None else v for v in valores], dtype=float)


def graficar_historia(historia: Historia, ax: plt.Axes, titulo: Optional[str] = None) -> None:
    """Dibuja las curvas de entrenamiento y validación contra millones de tokens vistos.

    Args:
        historia: Historia devuelta por ``entrenar``.
        ax: Ejes de matplotlib.
        titulo: Título opcional (por defecto, el nombre del modelo).
    """
    escala = historia['batch_efectivo'] * (CONTEXTO - 1) / 1e6
    tokens = np.array(historia['pasos']) * escala
    ax.plot(tokens, a_float(historia['train']), 'o-', color='#4C72B0', label='entrenamiento')
    ax.plot(tokens, a_float(historia['val']), 's-', color='#DD8452', label='validación')
    ax.axvline(historia['mejor_paso'] * escala, color='gray', ls=':', label=f'mejor paso ({historia["mejor_paso"]})')
    ax.set_xlabel('millones de tokens vistos')
    ax.set_ylabel('NLL por token (nats)')
    ax.set_title(titulo or historia['nombre'])
    ax.grid(alpha=0.3)
    ax.legend()

In [ ]:
@como_servicio
def entrenar_gpt_desde_cero() -> Historia:
    """Entrena el GPT desde cero (o reutiliza su checkpoint) y lo guarda con torch.save.

    Returns:
        Historia de entrenamiento del GPT.
    """
    if os.path.exists(RUTA_GPT) and not REENTRENAR:
        historia = torch.load(RUTA_GPT, map_location='cpu', weights_only=True)['historia']
        print(f'Checkpoint encontrado en {RUTA_GPT}: se reutiliza '
              f'(mejor paso {historia["mejor_paso"]}, val {historia["mejor_val"]:.4f}).')
        return historia
    fijar_semillas(SEMILLA)
    gpt = GPTDesdeCero(**GPT_CFG)
    historia = entrenar(gpt, 'GPT desde cero', **GPT_ENTRENO)
    torch.save({'config': GPT_CFG,
                'estado': {k: v.cpu() for k, v in gpt.state_dict().items()},
                'historia': historia}, RUTA_GPT)
    print('Checkpoint guardado en', RUTA_GPT)
    del gpt
    liberar_gpu()
    return historia


historia_gpt = entrenar_gpt_desde_cero().unwrap()

fig, ax = plt.subplots(figsize=(8, 3.8))
graficar_historia(historia_gpt, ax)
plt.tight_layout()
plt.show()

## 7. Finetuning de los modelos pre-entrenados

Finetuning **completo** (todos los parámetros se actualizan; no se usa LoRA/QLoRA) de SmolLM2-135M y SmolLM2-360M con `AutoModelForCausalLM`. Los modelos se entrenan de a uno: se guarda el checkpoint y se libera la GPU (`del modelo`, `torch.cuda.empty_cache()`) antes de cargar el siguiente. Técnicas de reducción de memoria del documento anexo:

- **Precisión mixta:** `contexto_amp()` (autocast fp16/bf16) y `GradScaler` dentro de `entrenar()`; pesos maestros en fp32.
- **Acumulación de gradiente:** `grad_accum` en `entrenar()`, batch efectivo de 16 en ambos modelos.
- **Gradient checkpointing:** `gradient_checkpointing_enable()` y `use_cache = False` en `finetunear()`.

| Modelo | lr (AdamW) | Épocas | Micro-batch × acumulación | Weight decay |
|---|---|---|---|---|
| SmolLM2-135M | 5e-5 | 1 | 4 × 4 | 0,01 |
| SmolLM2-360M | 3e-5 | 1 | 2 × 8 | 0,01 |

El paso 0 de cada curva corresponde a la validación del modelo pre-entrenado sin finetuning (baseline). Los checkpoints se guardan con `save_pretrained`, junto con el tokenizador y la historia de entrenamiento.

In [ ]:
for clave, nombre in MODELOS_HF.items():
    modelo = AutoModelForCausalLM.from_pretrained(nombre)
    cfg = modelo.config
    print(nombre)
    print('  Tamaño del vocabulario:', cfg.vocab_size)
    print(f'  Parámetros: {contar_parametros(modelo):,}')
    print(f'  Capas: {cfg.num_hidden_layers} | dimensión oculta: {cfg.hidden_size} | '
          f'cabezas de atención: {cfg.num_attention_heads} | contexto máximo: {cfg.max_position_embeddings}')
    assert cfg.vocab_size == tokenizer.vocab_size == GPT_CFG['vocab']
    del modelo
liberar_gpu()
print(f'\nComparación: el GPT desde cero tiene {GPT_CFG["n_capas"]} capas y dimensión {GPT_CFG["d_modelo"]}.')

In [ ]:
@como_servicio
def finetunear(clave: str) -> Historia:
    """Finetuning completo de un SmolLM2 (o reutilización de su checkpoint).

    Args:
        clave: '135M' o '360M'.

    Returns:
        Historia de entrenamiento; ``val[0]`` es la pérdida de validación del modelo pre-entrenado.
    """
    carpeta = CARPETA_FT[clave]
    ruta_historia = os.path.join(carpeta, 'historia.json')
    if os.path.exists(ruta_historia) and not REENTRENAR:
        with open(ruta_historia, encoding='utf-8') as f:
            historia = json.load(f)
        print(f'Checkpoint encontrado en {carpeta}: se reutiliza '
              f'(mejor paso {historia["mejor_paso"]}, val {historia["mejor_val"]:.4f}).')
        return historia

    fijar_semillas(SEMILLA)
    modelo = AutoModelForCausalLM.from_pretrained(MODELOS_HF[clave]).float()
    assert all(p.requires_grad for p in modelo.parameters()), 'El finetuning debe ser completo'
    modelo.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    modelo.config.use_cache = False

    historia = entrenar(modelo, f'SmolLM2-{clave} finetuned', **FT_ENTRENO[clave])

    modelo.config.use_cache = True
    modelo.save_pretrained(carpeta)
    tokenizer.save_pretrained(carpeta)
    with open(ruta_historia, 'w', encoding='utf-8') as f:
        json.dump(historia, f)
    print('Checkpoint guardado en', carpeta)
    del modelo
    liberar_gpu()
    return historia

In [ ]:
historia_135 = finetunear('135M').unwrap()

In [ ]:
historia_360 = finetunear('360M').unwrap()

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 3.8))
for ax, h in zip(ejes, (historia_135, historia_360)):
    graficar_historia(h, ax)
    ax.axhline(h['val'][0], color='#55A868', ls='--', label='baseline (sin finetuning)')
    ax.legend()
plt.tight_layout()
plt.show()

for h in (historia_135, historia_360):
    print(f'{h["nombre"]}: val {h["val"][0]:.4f} -> {h["mejor_val"]:.4f} '
          f'(PPL {math.exp(h["val"][0]):.2f} -> {math.exp(h["mejor_val"]):.2f}) | {h["segundos"] / 60:.1f} min'
          + (f' | memoria máxima {h["memoria_max_gb"]:.1f} GB' if h.get('memoria_max_gb') else ''))

## 8. Evaluación y comparación

Las cinco configuraciones se evalúan sobre los mismos 800 archivos de prueba (1.355 bloques de 256 tokens).

- **Perplejidad (métrica principal):** $\text{PPL} = \exp\left(-\frac{1}{N}\sum_{i=1}^{N}\log p(x_i \mid x_{<i})\right)$. Se acumula $-\log p$ sobre todos los tokens predichos y se divide una sola vez por $N$; no se promedian perplejidades por lote, no hay relleno y la pasada se hace en fp32.
- **Validez sintáctica (métrica secundaria):** 200 archivos de prueba elegidos con semilla fija; prompt de 64 tokens, continuación de hasta 128 tokens con `generar()` (temperatura 0,8, top-k 50) y verificación con `ast.parse` sin ejecutar el código. Si la generación se corta por el límite de tokens, se descarta la última línea antes de verificar. Se reporta el porcentaje con intervalo de Wilson al 95 % y se compara con una referencia: la continuación real del mismo archivo, cortada a la misma longitud.

In [ ]:
CONFIGURACIONES: Dict[str, str] = {
    'GPT desde cero': RUTA_GPT,
    'SmolLM2-135M pre-entrenado': MODELOS_HF['135M'],
    'SmolLM2-135M finetuned': CARPETA_FT['135M'],
    'SmolLM2-360M pre-entrenado': MODELOS_HF['360M'],
    'SmolLM2-360M finetuned': CARPETA_FT['360M'],
}
ENTRENADOS = ['GPT desde cero', 'SmolLM2-135M finetuned', 'SmolLM2-360M finetuned']


def cargar_configuracion(nombre: str) -> ModeloLM:
    """Carga una de las cinco configuraciones en DEVICE, en fp32 y en modo evaluación.

    Args:
        nombre: Clave de ``CONFIGURACIONES``.

    Returns:
        Modelo listo para inferencia.
    """
    ruta = CONFIGURACIONES[nombre]
    if ruta == RUTA_GPT:
        checkpoint = torch.load(ruta, map_location='cpu', weights_only=True)
        modelo = GPTDesdeCero(**checkpoint['config'])
        modelo.load_state_dict(checkpoint['estado'])
    else:
        modelo = AutoModelForCausalLM.from_pretrained(ruta).float()
        modelo.config.use_cache = False
    return modelo.to(DEVICE).eval()


def perplejidad(modelo: ModeloLM, bloques: torch.Tensor) -> Tuple[float, float]:
    """NLL promedio por token y perplejidad, calculadas en fp32.

    Args:
        modelo: Modelo a evaluar.
        bloques: Shape: (n_bloques, seq_len).

    Returns:
        Tupla (nll, ppl) con ppl = exp(nll).
    """
    nll = evaluar_nll(modelo, bloques, usar_amp=False)
    return nll, math.exp(nll)

In [ ]:
@torch.no_grad()
def generar(modelo: ModeloLM, prompts: torch.Tensor, max_nuevos: int, temperatura: float = TEMPERATURA,
            top_k: Optional[int] = TOP_K, semilla: int = SEMILLA) -> List[Tuple[List[int], bool]]:
    """Generación autorregresiva implementada desde cero (sin ``model.generate()``).

    En cada paso: logits de la última posición -> división por la temperatura -> filtro top-k ->
    softmax -> muestreo multinomial (o argmax si temperatura == 0) -> concatenación al contexto.
    La ventana se recorta a los últimos CONTEXTO tokens. Las secuencias que emiten EOS se congelan.

    Args:
        modelo: Cualquiera de las cinco configuraciones.
        prompts: Ids de los prompts, todos de la misma longitud. Shape: (batch_size, prompt_len).
        max_nuevos: Máximo de tokens a generar por secuencia.
        temperatura: Temperatura de muestreo; 0 equivale a decodificación voraz.
        top_k: Número de tokens candidatos que se conservan (None o 0 desactiva el filtro).
        semilla: Semilla del generador de muestreo.

    Returns:
        Por secuencia, una tupla (tokens_nuevos_sin_EOS, terminó_con_EOS).
    """
    modelo.eval()
    azar = torch.Generator().manual_seed(semilla)
    ids = prompts.to(DEVICE)
    terminado = torch.zeros(len(ids), dtype=torch.bool)
    nuevos: List[torch.Tensor] = []
    for _ in range(max_nuevos):
        ventana = ids[:, -CONTEXTO:]                                   # (B, <=CONTEXTO)
        with contexto_amp():
            logits = logits_de(modelo, ventana)[:, -1, :]              # (B, V)
        logits = logits.float().cpu()
        if temperatura == 0:
            siguiente = logits.argmax(dim=-1)
        else:
            logits = logits / temperatura
            if top_k:
                umbral = torch.topk(logits, top_k, dim=-1).values[:, -1:]
                logits = logits.masked_fill(logits < umbral, float('-inf'))
            probabilidades = F.softmax(logits, dim=-1)
            siguiente = torch.multinomial(probabilidades, 1, generator=azar).squeeze(1)  # (B,)
        siguiente[terminado] = EOS_ID
        terminado |= siguiente == EOS_ID
        nuevos.append(siguiente)
        ids = torch.cat([ids, siguiente.unsqueeze(1).to(DEVICE)], dim=1)
        if terminado.all():
            break

    salidas = []
    for fila in torch.stack(nuevos, dim=1).tolist():
        if EOS_ID in fila:
            salidas.append((fila[:fila.index(EOS_ID)], True))
        else:
            salidas.append((fila, False))
    return salidas


def es_valido(codigo: str, termino_en_eos: bool) -> Tuple[bool, Optional[str]]:
    """Verifica la validez sintáctica con ast.parse, sin ejecutar el código.

    Args:
        codigo: Texto completo (prompt + continuación).
        termino_en_eos: Si es False la generación se cortó por el límite y se descarta la última línea.

    Returns:
        Tupla (válido, motivo); motivo es None si el código es válido.
    """
    if not termino_en_eos:
        codigo = codigo.rsplit('\n', 1)[0]
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            ast.parse(codigo)
        return True, None
    except SyntaxError as error:
        return False, error.msg
    except (ValueError, RecursionError, MemoryError) as error:
        return False, type(error).__name__


def intervalo_wilson(aciertos: int, n: int, z: float = 1.96) -> Tuple[float, float]:
    """Intervalo de confianza de Wilson para una proporción (robusto con n pequeño).

    Args:
        aciertos: Número de éxitos.
        n: Número de ensayos.
        z: Cuantil normal (1.96 para 95 %).

    Returns:
        Tupla (límite_inferior, límite_superior) en [0, 1].
    """
    p = aciertos / n
    centro = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    margen = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centro - margen, centro + margen

In [ ]:
elegidos = np.random.default_rng(SEMILLA).choice(len(docs_test), size=N_VALIDEZ, replace=False)
assert all(len(docs_test[i]) > PROMPT_TOKENS for i in elegidos)
PROMPTS_VALIDEZ = torch.tensor([docs_test[i][:PROMPT_TOKENS] for i in elegidos])   # (N_VALIDEZ, PROMPT_TOKENS)


def validez_sintactica(salidas: List[Tuple[List[int], bool]]) -> List[Dict[str, Any]]:
    """Une cada prompt con su continuación, decodifica y verifica el texto con ast.parse.

    Args:
        salidas: Resultado de ``generar`` alineado con ``PROMPTS_VALIDEZ``.

    Returns:
        Una fila por prompt con el texto, si es válido, el motivo del error y si cerró con EOS.
    """
    filas = []
    for prompt, (nuevos, cerro) in zip(PROMPTS_VALIDEZ.tolist(), salidas):
        texto = tokenizer.decode(prompt + nuevos)
        valido, motivo = es_valido(texto, cerro)
        filas.append(dict(texto=texto, valido=valido, motivo=motivo, cerro=cerro))
    return filas


continuaciones_reales = []
for i in elegidos:
    real = docs_test[i][PROMPT_TOKENS:PROMPT_TOKENS + NUEVOS_TOKENS]
    if EOS_ID in real:
        continuaciones_reales.append((real[:real.index(EOS_ID)], True))
    else:
        continuaciones_reales.append((real, False))
filas_ref = validez_sintactica(continuaciones_reales)
k_ref = sum(f['valido'] for f in filas_ref)
print(f'{N_VALIDEZ} prompts de {PROMPT_TOKENS} tokens tomados de {len(docs_test)} archivos de prueba')
print(f'Referencia (código real cortado a {NUEVOS_TOKENS} tokens): {k_ref}/{N_VALIDEZ} = {k_ref / N_VALIDEZ:.1%} válidos')

In [ ]:
@como_servicio
def evaluar_configuracion(nombre: str) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:
    """Calcula la perplejidad de prueba y la validez sintáctica de una configuración.

    Args:
        nombre: Clave de ``CONFIGURACIONES``.

    Returns:
        Tupla (métricas, filas_de_validez).
    """
    t0 = time.time()
    modelo = cargar_configuracion(nombre)
    nll, ppl = perplejidad(modelo, bloques_test)
    salidas = []
    for i in range(0, N_VALIDEZ, LOTE_GEN):
        salidas += generar(modelo, PROMPTS_VALIDEZ[i:i + LOTE_GEN], NUEVOS_TOKENS, semilla=SEMILLA + i)
    filas = validez_sintactica(salidas)
    validos = sum(f['valido'] for f in filas)
    metricas = dict(parametros=contar_parametros(modelo), nll=nll, ppl=ppl, validos=validos, n=N_VALIDEZ,
                    segundos_eval=time.time() - t0)
    del modelo
    liberar_gpu()
    return metricas, filas


if os.path.exists(RUTA_RESULTADOS) and not (REENTRENAR or REEVALUAR):
    with open(RUTA_RESULTADOS, encoding='utf-8') as f:
        guardado = json.load(f)
    resultados, filas_validez = guardado['resultados'], guardado['filas_validez']
    print(f'Resultados de prueba leídos de {RUTA_RESULTADOS} (use REEVALUAR = True para recalcular).')
else:
    resultados, filas_validez = {}, {}
    for nombre in CONFIGURACIONES:
        resultados[nombre], filas_validez[nombre] = evaluar_configuracion(nombre).unwrap()
    with open(RUTA_RESULTADOS, 'w', encoding='utf-8') as f:
        json.dump(dict(resultados=resultados, filas_validez=filas_validez,
                       referencia=dict(validos=k_ref, n=N_VALIDEZ)), f, indent=1)

for nombre, r in resultados.items():
    print(f'{nombre:<28} | NLL {r["nll"]:.4f} | PPL {r["ppl"]:10.2f} | válidas {r["validos"]:3d}/{r["n"]} '
          f'| {r["segundos_eval"]:4.0f} s')

In [ ]:
HISTORIAS: Dict[str, Historia] = {'GPT desde cero': historia_gpt,
                                  'SmolLM2-135M finetuned': historia_135,
                                  'SmolLM2-360M finetuned': historia_360}

filas_tabla = []
for nombre, r in resultados.items():
    bajo, alto = intervalo_wilson(r['validos'], r['n'])
    historia = HISTORIAS.get(nombre)
    filas_tabla.append({
        'configuración': nombre,
        'parámetros (M)': round(r['parametros'] / 1e6, 1),
        'NLL prueba': round(r['nll'], 4),
        'PPL prueba': round(r['ppl'], 2),
        'válidas (%)': round(100 * r['validos'] / r['n'], 1),
        'IC 95 % (Wilson)': f'{100 * bajo:.1f} - {100 * alto:.1f}',
        'entrenamiento (min)': round(historia['segundos'] / 60, 1) if historia else '-',
        'memoria máx. (GB)': round(historia['memoria_max_gb'], 1) if historia and historia.get('memoria_max_gb') else '-',
    })
bajo, alto = intervalo_wilson(k_ref, N_VALIDEZ)
filas_tabla.append({'configuración': 'Referencia: código real', 'parámetros (M)': '-', 'NLL prueba': '-',
                    'PPL prueba': '-', 'válidas (%)': round(100 * k_ref / N_VALIDEZ, 1),
                    'IC 95 % (Wilson)': f'{100 * bajo:.1f} - {100 * alto:.1f}', 'entrenamiento (min)': '-',
                    'memoria máx. (GB)': '-'})
tabla = pd.DataFrame(filas_tabla).set_index('configuración')
display(tabla)

In [ ]:
nombres = list(resultados)
colores = ['#8172B3', '#9ecae1', '#3182bd', '#fdae6b', '#e6550d']
etiquetas = [n.replace('SmolLM2-', '').replace(' pre-entrenado', '\npre-entrenado').replace(' finetuned', '\nfinetuned')
             .replace('GPT desde cero', 'GPT\ndesde cero') for n in nombres]

fig, ejes = plt.subplots(1, 2, figsize=(13, 4.2))
ppls = [resultados[n]['ppl'] for n in nombres]
ejes[0].bar(range(len(nombres)), ppls, color=colores)
for j, valor in enumerate(ppls):
    ejes[0].text(j, valor, f'{valor:.2f}', ha='center', va='bottom', fontsize=9)
ejes[0].set_yscale('log')
ejes[0].set_ylabel('perplejidad en prueba (escala log)')
ejes[0].set_title('Perplejidad (menor es mejor)')

porcentajes, errores = [], [[], []]
for n in nombres:
    p = resultados[n]['validos'] / resultados[n]['n']
    bajo, alto = intervalo_wilson(resultados[n]['validos'], resultados[n]['n'])
    porcentajes.append(100 * p)
    errores[0].append(100 * max(p - bajo, 0.0))
    errores[1].append(100 * max(alto - p, 0.0))
ejes[1].bar(range(len(nombres)), porcentajes, yerr=errores, capsize=4, color=colores)
ejes[1].axhline(100 * k_ref / N_VALIDEZ, color='gray', ls='--', label='código real')
ejes[1].set_ylim(0, 100)
ejes[1].set_ylabel('% de textos que ast.parse acepta')
ejes[1].set_title(f'Validez sintáctica ({N_VALIDEZ} prompts, IC 95 %)')
ejes[1].legend(loc='upper left')

for ax in ejes:
    ax.set_xticks(range(len(nombres)))
    ax.set_xticklabels(etiquetas, fontsize=9)
    ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.4))
colores_curvas = {'GPT desde cero': '#8172B3', 'SmolLM2-135M finetuned': '#3182bd', 'SmolLM2-360M finetuned': '#e6550d'}
techo = []
for nombre, h in HISTORIAS.items():
    tokens = np.array(h['pasos']) * h['batch_efectivo'] * (CONTEXTO - 1) / 1e6
    ax.plot(tokens, a_float(h['val']), 's-', color=colores_curvas[nombre], label=f'{nombre} (validación)')
    ax.plot(tokens, a_float(h['train']), 'o--', color=colores_curvas[nombre], alpha=0.45, label=f'{nombre} (entrenamiento)')
    techo.append(np.nanmax(a_float(h['val'])[1:]))
for h, nombre in ((historia_135, 'SmolLM2-135M'), (historia_360, 'SmolLM2-360M')):
    ax.axhline(h['val'][0], color=colores_curvas[f'{nombre} finetuned'], ls=':', alpha=0.8,
               label=f'{nombre} pre-entrenado (baseline)')
ax.set_ylim(0, max(techo) * 1.1)
ax.set_xlabel('millones de tokens vistos')
ax.set_ylabel('NLL por token (nats)')
ax.set_title('Curvas de pérdida de los tres modelos entrenados')
ax.grid(alpha=0.3)
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
print('Errores de sintaxis más frecuentes por configuración\n')
for nombre, filas in list(filas_validez.items()) + [('Referencia: código real', filas_ref)]:
    motivos = Counter(f['motivo'] for f in filas if not f['valido'])
    cerradas = sum(f['cerro'] for f in filas)
    print(f'{nombre} | válidas {sum(f["valido"] for f in filas)}/{len(filas)} | cerraron con EOS {cerradas}/{len(filas)}')
    for motivo, n in motivos.most_common(3):
        print(f'    {n:3d} x {motivo}')

In [ ]:
def cambio_ppl(antes: str, despues: str) -> Tuple[float, float, float]:
    """Perplejidades de dos configuraciones y su cambio relativo en %.

    Args:
        antes: Configuración de referencia.
        despues: Configuración comparada.

    Returns:
        Tupla (ppl_antes, ppl_despues, cambio_porcentual).
    """
    a, b = resultados[antes]['ppl'], resultados[despues]['ppl']
    return a, b, 100 * (b / a - 1)


def cambio_validez(antes: str, despues: str) -> Tuple[float, float, bool]:
    """Porcentajes de validez de dos configuraciones y si sus IC de Wilson se traslapan.

    Args:
        antes: Configuración de referencia.
        despues: Configuración comparada.

    Returns:
        Tupla (pct_antes, pct_despues, intervalos_se_traslapan).
    """
    ra, rb = resultados[antes], resultados[despues]
    ia, ib = intervalo_wilson(ra['validos'], ra['n']), intervalo_wilson(rb['validos'], rb['n'])
    return 100 * ra['validos'] / ra['n'], 100 * rb['validos'] / rb['n'], ia[0] <= ib[1] and ib[0] <= ia[1]


def fmt(valor: float, decimales: int = 2) -> str:
    """Formatea un número con coma decimal.

    Args:
        valor: Número a formatear.
        decimales: Cifras decimales.

    Returns:
        Texto con coma decimal.
    """
    return f'{valor:.{decimales}f}'.replace('.', ',')


lineas = ['### Análisis comparativo (generado a partir de los resultados de esta ejecución)', '']

partes = []
for clave in MODELOS_HF:
    a, b, d = cambio_ppl(f'SmolLM2-{clave} pre-entrenado', f'SmolLM2-{clave} finetuned')
    va, vb, solapa = cambio_validez(f'SmolLM2-{clave} pre-entrenado', f'SmolLM2-{clave} finetuned')
    partes.append(f'en el {clave} la PPL pasa de {fmt(a)} a {fmt(b)} ({fmt(d, 1)} %) y la validez de '
                  f'{fmt(va, 1)} % a {fmt(vb, 1)} % ({"IC traslapados: diferencia no concluyente" if solapa else "IC sin traslape"})')
lineas.append('- **Baseline frente a finetuning.** ' + '; '.join(partes) + '.')

a, b, d = cambio_ppl('SmolLM2-135M pre-entrenado', 'SmolLM2-360M pre-entrenado')
a2, b2, d2 = cambio_ppl('SmolLM2-135M finetuned', 'SmolLM2-360M finetuned')
cruce = resultados['SmolLM2-360M pre-entrenado']['ppl'] < resultados['SmolLM2-135M finetuned']['ppl']
lineas.append(f'- **135M frente a 360M.** El modelo grande reduce la PPL un {fmt(-d, 1)} % sin finetuning '
              f'({fmt(a)} -> {fmt(b)}) y un {fmt(-d2, 1)} % con finetuning ({fmt(a2)} -> {fmt(b2)}). '
              + ('El 360M pre-entrenado ya supera al 135M finetuned: la capacidad del modelo pesa más que la '
                 'especialización con este volumen de datos. ' if cruce else
                 'El 135M finetuned supera al 360M pre-entrenado: la especialización compensa la menor capacidad. ')
              + 'La ganancia relativa del finetuning es '
              + ('menor' if cambio_ppl('SmolLM2-360M pre-entrenado', 'SmolLM2-360M finetuned')[2]
                 > cambio_ppl('SmolLM2-135M pre-entrenado', 'SmolLM2-135M finetuned')[2] else 'mayor')
              + ' en el modelo grande, que parte de una distribución más cercana al código.')

mejor = min(resultados, key=lambda n: resultados[n]['ppl'])
a, b, d = cambio_ppl('GPT desde cero', mejor)
va, vb, solapa = cambio_validez('GPT desde cero', mejor)
lineas.append(f'- **Desde cero frente a pre-entrenados.** El GPT desde cero alcanza PPL {fmt(a)} frente a {fmt(b)} '
              f'del mejor modelo ({mejor}), y validez {fmt(va, 1)} % frente a {fmt(vb, 1)} % '
              f'({"IC traslapados" if solapa else "IC sin traslape"}). Con ≈ {len(bloques_train) * (CONTEXTO - 1) / 1e6:.1f} M '
              'de tokens de entrenamiento, el modelo desde cero aprende la sintaxis local (indentación, paréntesis, '
              'palabras clave) pero no la semántica de APIs que los SmolLM2 adquirieron en su pre-entrenamiento.')

partes = []
for nombre, h in HISTORIAS.items():
    j = h['pasos'].index(h['mejor_paso'])
    brecha = (h['val'][j] - h['train'][j]) if h['train'][j] is not None else float('nan')
    partes.append(f'{nombre}: mejor paso {h["mejor_paso"]} de {h["pasos"][-1]}, '
                  f'val final - mejor val = {h["val"][-1] - h["mejor_val"]:+.4f}, brecha val - train = {brecha:+.4f}')
lineas.append('- **Sobreajuste.** ' + '; '.join(partes) + '. Medidas aplicadas: dropout 0,1 y weight decay 0,1 en el '
              'GPT; una sola época y tasas de aprendizaje bajas con decaimiento coseno en el finetuning; validación '
              '4 veces por época y restauración del mejor checkpoint en los tres modelos. El último punto de la curva '
              'de entrenamiento de los finetuned promedia pocos pasos y es más ruidoso.')

lineas.append(f'- **Validez sintáctica.** La referencia de código real alcanza {fmt(100 * k_ref / N_VALIDEZ, 1)} %, '
              'porque cortar a 128 tokens deja con frecuencia paréntesis, corchetes o cadenas abiertas que la '
              'eliminación de la última línea no resuelve. Por eso los porcentajes deben leerse en relación con '
              'esa referencia y no contra 100 %. Las conclusiones cuantitativas se apoyan en la perplejidad.')
display(Markdown('\n'.join(lineas)))

## 9. Generación de ejemplos

Se usa `generar()` de la sección 8: bucle autorregresivo escrito a mano (logits de la última posición → temperatura → top-k → softmax → muestreo multinomial → concatenación al contexto), sin `model.generate()`. Tres prompts escritos a mano (función con docstring, clase con constructor y script con imports), con los mismos parámetros y semillas para cada modelo, de modo que las diferencias se deben solo al modelo. Primero se muestran los tres modelos entrenados por el equipo y, como contraste, los dos pre-entrenados. Cada ejemplo indica si `ast.parse` lo acepta.

In [ ]:
PROMPTS_EJEMPLO = [
    'def fibonacci(n):\n    """Return the n-th Fibonacci number."""',
    'class Stack(object):\n    """A simple LIFO stack."""\n\n    def __init__(self):',
    'import os\nimport sys\n\n\ndef read_lines(path):',
]
MOSTRAR_BASELINES = True


def mostrar_generacion(nombre: str, ids_prompt: List[int], nuevos: List[int], cerro: bool) -> None:
    """Muestra una generación como bloque de código con su verificación sintáctica.

    Args:
        nombre: Título del ejemplo.
        ids_prompt: Ids del prompt.
        nuevos: Ids generados (sin EOS).
        cerro: Si la generación terminó con EOS.
    """
    texto = tokenizer.decode(ids_prompt + nuevos)
    valido, motivo = es_valido(texto, cerro)
    estado = 'VÁLIDO según `ast.parse`' if valido else f'INVÁLIDO según `ast.parse` ({motivo})'
    final = 'cerró con EOS' if cerro else f'se cortó en {NUEVOS_EJEMPLO} tokens (la última línea no se valida)'
    display(Markdown(f'**{nombre}** · {estado} · {final}\n\n```python\n{texto}\n```'))

In [ ]:
modelos_ejemplo = ENTRENADOS + (['SmolLM2-135M pre-entrenado', 'SmolLM2-360M pre-entrenado'] if MOSTRAR_BASELINES else [])
ids_prompts = [tokenizer(p, add_special_tokens=False)['input_ids'] for p in PROMPTS_EJEMPLO]

for nombre in modelos_ejemplo:
    display(Markdown(f'---\n#### {nombre}'))
    modelo = cargar_configuracion(nombre)
    for j, ids_prompt in enumerate(ids_prompts, start=1):
        (nuevos, cerro), = generar(modelo, torch.tensor([ids_prompt]), NUEVOS_EJEMPLO, semilla=SEMILLA + j)
        mostrar_generacion(f'Ejemplo {j}', ids_prompt, nuevos, cerro)
    del modelo
    liberar_gpu()

## 10. Conclusiones

In [ ]:
ranking = sorted(resultados, key=lambda n: resultados[n]['ppl'])
texto_ranking = ' < '.join(f'{n} ({fmt(resultados[n]["ppl"])})' for n in ranking)
tiempos = ', '.join(f'{n}: {fmt(h["segundos"] / 60, 1)} min'
                    + (f' y {fmt(h["memoria_max_gb"], 1)} GB' if h.get('memoria_max_gb') else '')
                    for n, h in HISTORIAS.items())
descartes = sum(h.get('descartados', 0) for h in HISTORIAS.values())
d135 = cambio_ppl('SmolLM2-135M pre-entrenado', 'SmolLM2-135M finetuned')[2]
d360 = cambio_ppl('SmolLM2-360M pre-entrenado', 'SmolLM2-360M finetuned')[2]
display(Markdown('\n'.join([
    f'- **Ranking por perplejidad de prueba:** {texto_ranking}.',
    f'- **Efecto del finetuning:** reduce la PPL un {fmt(-d135, 1)} % en el 135M y un {fmt(-d360, 1)} % en el 360M.',
    f'- **Costo computacional del entrenamiento:** {tiempos}.'
    + (f' Se descartaron {descartes} micro-lotes por desborde numérico en fp16.' if descartes else ''),
])))

- **Pre-entrenamiento.** Es el factor dominante del desempeño. Con un corpus de 8.000 archivos (≈ 2,8 M de tokens de entrenamiento), el GPT desde cero aprende patrones superficiales del lenguaje (indentación, cierre de bloques, palabras clave, comentarios de licencia) pero su perplejidad queda un orden de magnitud por encima de los SmolLM2, que ya traen conocimiento general de código y de inglés.
- **Finetuning completo.** Especializa a ambos SmolLM2 en la distribución de `codeparrot-clean` con una sola época y reduce de forma consistente la perplejidad en prueba. Las técnicas de memoria (precisión mixta, acumulación de gradiente y gradient checkpointing) permiten entrenar el 360M completo en una GPU de 16 GB; el uso de fp16 en la T4 exige `GradScaler` y control de lotes no finitos.
- **Tamaño del modelo.** El 360M supera al 135M tanto antes como después del finetuning, a costa de aproximadamente el doble de tiempo y memoria. La ganancia relativa del finetuning tiende a ser menor en el modelo grande, que parte de una distribución más cercana al código.
- **Sobreajuste.** Las curvas de validación descienden hasta el final y la brecha entre validación y entrenamiento es pequeña en los tres modelos. La selección del mejor checkpoint por validación actúa como salvaguarda; con más épocas sobre el mismo corpus sería esperable que el GPT desde cero y, sobre todo, los SmolLM2 empezaran a sobreajustar.
- **Validez sintáctica.** Es una métrica secundaria y ruidosa: 200 muestras producen intervalos de confianza de ±7 puntos, el corte a 128 tokens deja estructuras abiertas incluso en el código real, y el corpus contiene código Python 2 (por ejemplo `print` sin paréntesis) que `ast.parse` rechaza. No mide si el código es correcto ni si ejecuta.
- **Limitaciones.** El contexto de 256 tokens solo cubre una fracción de cada archivo (98 % de los archivos lo exceden), por lo que los modelos no ven dependencias de largo alcance (imports usados mucho después, por ejemplo). Se usó una sola semilla, un único conjunto de hiperparámetros por modelo y un corpus pequeño, de modo que diferencias pequeñas podrían cambiar con otra semilla. La generación no usa caché de claves/valores, lo que la hace más lenta pero no afecta los resultados.
- **Trabajo futuro.** Explorar finetuning eficiente (LoRA/QLoRA) para comparar costo y desempeño frente al finetuning completo, evaluar con métricas funcionales (pass@k sobre pruebas unitarias) y ampliar el contexto para capturar archivos completos.